In [1]:
import polars as pl
import psycopg2
import dagster as dg

In [2]:
class PostgresResource():
    """Configuration schema for PostgreSQL resource."""
    def __init__(self, host: str, port: int, database: str, user: str, password: str, chunk_size: int):
        self.host: str = host
        self.port: int = port
        self.database: str = database
        self.user: str = user
        self.password: str = password
        self.chunk_size: int = chunk_size | 1_000_000
        
    def _get_connection(self) -> psycopg2.extensions.connection:
            """Creates a psycopg2 connection to PostgreSQL."""
            try:
                conn = psycopg2.connect(
                    dbname=self.database,
                    user=self.user,
                    host=self.host,
                    password=self.password,
                    port=self.port
                )
                return conn
            except Exception as e:
                print(f"Unable to connect to the database: {str(e)}")
                raise
    
    def _get_connection_string(self) -> str:
        """Constructs a PostgreSQL connection string."""
        return f"postgresql://{self.user}:{self.password}@{self.host}:{self.port}/{self.database}"

    def empty_table(self, table_name: str, schema: str) -> None:
        """Truncate the specified table."""
        conn = self._get_connection()
                
        try:
            with conn.cursor() as cur:
                full_table_name = f"{schema}.{table_name}"
                cur.execute(f"TRUNCATE TABLE {full_table_name} CASCADE")
                conn.commit()
        except Exception as e:
            print(f"Failed to truncate table: {str(e)}")
            raise
        finally:
            conn.close()
    
    def get_table_columns(self, table_name: str, schema: str) -> list:
        """Get column names of the specified table."""
        conn = self._get_connection()
        try:
            with conn.cursor() as cur:
                cur.execute(
                    """
                    SELECT column_name
                    FROM information_schema.columns
                    where 
                        table_schema='{schema}'
                        and
                        table_name='{table_name}'
                    order by ordinal_position;
                    """
                    
                )
                columns = [row[0] for row in cur.fetchall()]
                return columns
        except Exception as e:
            print(f"Failed to get table columns: {str(e)}")
            raise
        finally:
            conn.close()

    def _check_table_columns(self, table_name: str, schema: str, df: pl.DataFrame) -> None:
        """Check if DataFrame columns match the target table columns."""
        table_columns = self.get_table_columns(table_name, schema)
        df_columns = df.columns
        
        missing_columns = set(table_columns) - set(df_columns)
        extra_columns = set(df_columns) - set(table_columns)
        
        if missing_columns:
            raise ValueError(f"Missing columns in DataFrame: {missing_columns}")
        if extra_columns:
            raise ValueError(f"Extra columns in DataFrame: {extra_columns}")
    
    def load_polars_dataframe(self, df: pl.DataFrame, table_name: str, schema: str) -> None:
        """Load a Polars DataFrame to PostgreSQL in chunks."""
        try:
            self.empty_table(table_name, schema)
            total_chunks = (df.height + self.chunk_size - 1) // self.chunk_size

            connection_string = self._get_connection_string()

            for offset in range(0, df.height, self.chunk_size):
                message = f"Loading chunk {offset // self.chunk_size + 1} of {total_chunks}..."
                print(message)
                batch = df.slice(offset, self.chunk_size)
                batch.write_database(
                    table_name=f"{schema}.{table_name}",
                    if_table_exists="append",
                    connection=connection_string
                )
        except Exception as e:
            print(f"Failed to load data: {str(e)}")
            raise

    def get_query_results(self, query: str) -> pl.DataFrame:
        """Execute a SQL query and return results as a Polars DataFrame."""
        conn = self._get_connection()
        try:
            with conn.cursor() as cur:
                cur.execute(query)
                columns = [desc[0] for desc in cur.description]
                data = cur.fetchall()
                return pl.DataFrame(data, schema=columns)
        except Exception as e:
            print(f"Failed to execute query: {str(e)}")
            raise
        finally:
            conn.close()


In [3]:

POSTGRES_USER="dagster"
POSTGRES_PASSWORD="4g4WF~E*Zsl217aSErNKu9olVTuP=,HO]dP/FAlH"
POSTGRES_DATABASE="dagster"
POSTGRES_HOST="localhost"
POSTGRES_PORT=5432
IMDB_SCHEMA="imdb"

test = PostgresResource(
    host=POSTGRES_HOST,
    port=POSTGRES_PORT,
    database=POSTGRES_DATABASE,
    user=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
    chunk_size=1_000_000
)

In [4]:
path = "/media/user/Data/dirkv/Code/dagster/dagster-workspace/data/imdb/inputs/imdb_files/title.ratings.tsv"
df = pl.read_csv(path, separator="\t")
df = pl.read_csv(
        path,
        has_header=True,
        separator="\t",
        truncate_ragged_lines=True,
        null_values="\\N",
        quote_char=None,
        schema={
            "tconst": pl.Utf8,
            "averageRating": pl.Float16,
            "numVotes": pl.UInt32,
        },
    )
df = df.rename({"averageRating": "average_rating", "numVotes":"num_votes"})
# test.empty_table("title_ratings", IMDB_SCHEMA)
test.load_polars_dataframe(df, "title_ratings", IMDB_SCHEMA)

FileNotFoundError: No such file or directory (os error 2): .../Data/dirkv/Code/dagster/dagster-workspace/data/imdb/inputs/imdb_files/title.ratings.tsv (set POLARS_VERBOSE=1 to see full path)

In [ ]:
results = test.get_query_results(f"SELECT * FROM {IMDB_SCHEMA}.title_ratings LIMIT 5")
results

## read google sheets file

In [2]:
import os.path
import pathlib

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

import gspread
import polars as pl

In [ ]:
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
    

"""Shows basic usage of the Drive v3 API.
Prints the names and ids of the first 10 files the user has access to.
"""
creds = None
# The file token.json stores the user's access and refresh tokens, and is
# created automatically when the authorization flow completes for the first
# time.
if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file("token.json", SCOPES)
# If there are no (valid) credentials available, let the user log in.
if not creds or not creds.valid:
    # print(
    #     pathlib.Path(__file__).parent.parent.parent.resolve()
    #     / "google_api_credentials.json"
    # )
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            "google_api_credentials.json", SCOPES
        )

    creds = flow.run_local_server(port=0)
    # Save the credentials for the next run
    with open("token.json", "w") as token:
        token.write(creds.to_json())

try:
    gc = gspread.authorize(creds)

    sheet = gc.open_by_key("1TD_zFb5lqa4-7hAjepTfIbUVRKDvuL_PS4WAE_Am3Ek")

    gekeken_ws = sheet.worksheet("gekeken")
    nog_kijken_ws = sheet.worksheet("nog kijken")

    df_gekeken = pl.DataFrame(gekeken_ws.get_all_records())
    df_gekeken = df_gekeken.with_columns(
        pl.col("date").str.strptime(pl.Date, format="%d/%m/%Y").alias("date")
    )
    df_nog_kijken = pl.DataFrame(nog_kijken_ws.get_all_records())

    print(df_gekeken)
    print()
    print(df_nog_kijken)
except HttpError as error:
    # TODO(developer) - Handle errors from drive API.
    print(f"An error occurred: {error}")

shape: (29, 6)
┌────────────┬────────────┬───────────────────────────────┬────────────┬───────────┬─────────┐
│ toegevoegd ┆ tconst     ┆ titel                         ┆ date       ┆ enjoyment ┆ quality │
│ ---        ┆ ---        ┆ ---                           ┆ ---        ┆ ---       ┆ ---     │
│ str        ┆ str        ┆ str                           ┆ str        ┆ f64       ┆ f64     │
╞════════════╪════════════╪═══════════════════════════════╪════════════╪═══════════╪═════════╡
│            ┆ tt39316472 ┆ jackass b&l                   ┆ 24/07/2026 ┆ 2.0       ┆ 2.0     │
│            ┆ tt14173636 ┆ invite                        ┆ 22/07/2026 ┆ 1.0       ┆ 2.0     │
│            ┆ tt33764258 ┆ odissy                        ┆ 17/07/2026 ┆ 3.0       ┆ 3.0     │
│            ┆ tt37163509 ┆ la venus electric             ┆ 15/07/2026 ┆ 3.0       ┆ 2.0     │
│            ┆ tt0120669  ┆ fear and loathing             ┆ 10/07/2026 ┆ 4.0       ┆ 2.5     │
│ …          ┆ …          ┆ …      

# add data to database

In [7]:
import psycopg2
from psycopg2.extras import execute_values

In [28]:
POSTGRES_USER="dagster"
POSTGRES_PASSWORD="4g4WF~E*Zsl217aSErNKu9olVTuP=,HO]dP/FAlH"
POSTGRES_DATABASE="dagster"
POSTGRES_HOST="localhost"
POSTGRES_PORT=5432
IMDB_SCHEMA="imdb"


conn = psycopg2.connect(f"dbname={POSTGRES_DATABASE} user={POSTGRES_USER} password={POSTGRES_PASSWORD} host={POSTGRES_HOST}")
cur = conn.cursor()

# 1) gekeken films toevoegen aan db of updaten naar gekeken
watched_tconsts = df_gekeken["tconst"].to_list()
sql = """
INSERT INTO imdb.watch_status (tconst, watched, priority)
VALUES (%s, TRUE, FALSE)
ON CONFLICT (tconst)
DO UPDATE 
SET watched = TRUE;
"""

cur.executemany(sql, [(t,) for t in watched_tconsts])
conn.commit()


# 2) ongekeken films toevoegen aan db
# Convert Polars DataFrame to list of tuples
df_nog_kijken = df_nog_kijken.with_columns([
    pl.when(pl.col("netflix") == "")
      .then(None)
      .otherwise(pl.col("netflix"))
      .alias("netflix"),

    pl.when(pl.col("prime") == "")
      .then(None)
      .otherwise(pl.col("prime"))
      .alias("prime"),

    pl.lit(False).alias("watched")
])

rows = list(
    zip(
        df_nog_kijken["tconst"].to_list(),
        df_nog_kijken["watched"].to_list(),
        df_nog_kijken["priority"].to_list(),
        df_nog_kijken["netflix"].to_list(),
        df_nog_kijken["prime"].to_list(),
    )
)

sql = """
INSERT INTO imdb.watch_status (tconst, watched, priority, netflix, prime)
VALUES %s
ON CONFLICT (tconst)
DO UPDATE SET
    watched  = imdb.watch_status.watched,
    priority = EXCLUDED.priority,
    netflix  = COALESCE(imdb.watch_status.netflix, EXCLUDED.netflix),
    prime    = COALESCE(imdb.watch_status.prime,   EXCLUDED.prime);
"""

execute_values(cur, sql, rows)
conn.commit()


# 3) reviews toevoegen aan database
rows = list(
    zip(
        df_gekeken["tconst"].to_list(),
        df_gekeken["date"].to_list(),
        df_gekeken["enjoyment"].rename("enjoyment_score").to_list(),
        df_gekeken["quality"].rename("quality_score").to_list(),
    )
)

sql = """
INSERT INTO imdb.watch_date_scores (tconst, date, enjoyment_score, quality_score)
VALUES %s
ON CONFLICT (tconst, date, enjoyment_score, quality_score) DO NOTHING;
"""

execute_values(cur, sql, rows)
conn.commit()

cur.close()
conn.close()

## check if some movies did not get added and update the sheet

## copy sheets and graphs to google drive